# Przygotowanie danych

In [1]:
import pandas as pd
import numpy as np
import os
import glob
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

print("="*80)
print(">>> KROK 1: ŁADOWANIE PRÓBKI DANYCH I INŻYNIERIA CECH <<<")
print("="*80)

ALL_FILES_PATTERN = "../dane/treningowe/*_netflow-extended.csv" 
RANDOM_SEED = 42

def load_stratified_data(file_pattern, sample_size):
    files = glob.glob(file_pattern)
    dfs = []
    print(f"[*] Pobieranie próbek z {len(files)} plików...")
    for path in files:
        try:
            df_temp = pd.read_csv(path, on_bad_lines="skip", low_memory=False)
            # Losujemy próbkę
            df_sampled = df_temp.sample(n=sample_size, random_state=RANDOM_SEED) if len(df_temp) > sample_size else df_temp
            df_sampled['original_file'] = os.path.basename(path)
            dfs.append(df_sampled)
        except Exception as e: 
            print(f"  [ERR] {path}: {e}")
            
    if not dfs: return pd.DataFrame()
    
    final_df = pd.concat(dfs, ignore_index=True).dropna(axis=1, how='all')
    if 'StartTime' in final_df.columns: 
        print(f"[*] Konwersja czasu i sortowanie...")
        final_df['StartTime'] = pd.to_datetime(final_df['StartTime'], errors='coerce')
        final_df.sort_values(by='StartTime', inplace=True, na_position='last')
    
    return final_df

# Tu ustawiamy duże sample_size, np. 30000 z każdego pliku treningowego
df_train = load_stratified_data(ALL_FILES_PATTERN, sample_size=2000)

NUM_COLS = ['Dur', 'TotPkts', 'SrcPkts', 'DstPkts', 'TotBytes', 'SrcBytes', 'DstBytes', 
            'Bytes_per_Pkt', 'Pkts_Ratio', 'Pkts_Freq', 'Bytes_Ratio'] 

BIN_COLS = ['argus_src_loss', 'argus_dst_loss', 'argus_encap', 'argus_mismatch',
            'is_well_known_port', 'is_ephemeral_port'] 

TARGET_PORTS = [
    21, 22, 23, 80, 443, 8080, 8081, 8888, 8889, 3389, 5900, 
    25, 53, 110, 143, 1433, 3306, 5432, 6379, 27017,
    139, 445, 
    2323, 5060, 5061, 3128, 1080, 53413, 37777,
    19, 111, 123, 500, 623, 1194, 1900, 4500, 5353, 5683
]

def clean_and_engineer_train(df, valid_states, valid_protos):
    df = df.copy()
    for c in ['Sport', 'Dport', 'TotBytes', 'TotPkts', 'SrcPkts', 'DstPkts', 'SrcBytes', 'DstBytes', 'Dur']:
        if c in df.columns: 
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
            
    df['Bytes_per_Pkt'] = df['TotBytes'] / (df['TotPkts'] + 1e-6)
    df['Pkts_Ratio'] = df['SrcPkts'] / (df['DstPkts'] + 1e-6)
    df['Pkts_Freq'] = df['TotPkts'] / (df['Dur'] + 1e-6)
    df['Bytes_Ratio'] = df['SrcBytes'] / (df['DstBytes'] + 1e-6)
    df['is_well_known_port'] = ((df['Dport'] > 0) & (df['Dport'] <= 1024)).astype(int)
    df['is_ephemeral_port'] = (df['Dport'] >= 49152).astype(int)

    for p in TARGET_PORTS: 
        df[f'is_dport_{p}'] = (df['Dport'] == p).astype(int)
        
    if 'Flgs' in df.columns:
        flgs = df['Flgs'].astype(str).str.lower()
        df['argus_src_loss'] = flgs.apply(lambda x: 1 if 's' in x else 0)
        df['argus_dst_loss'] = flgs.apply(lambda x: 1 if 'd' in x else 0)
        df['argus_encap']    = flgs.apply(lambda x: 1 if 'e' in x else 0)
        df['argus_mismatch'] = flgs.apply(lambda x: 1 if '*' in x else 0)
    else:
        for c in BIN_COLS: df[c] = 0
            
    if 'State' in df.columns:
        df['State'] = df['State'].astype(str).str.upper()
        df.loc[~df['State'].isin(valid_states), 'State'] = 'OTHER'
            
    if 'Proto' in df.columns:
        df['Proto'] = df['Proto'].astype(str).str.lower()
        df.loc[~df['Proto'].isin(valid_protos), 'Proto'] = 'OTHER'

    return df

print("[*] Generowanie profili behawioralnych...")
top_states = df_train['State'].astype(str).str.upper().value_counts().nlargest(20).index.tolist()
top_protos = df_train['Proto'].astype(str).str.lower().value_counts().nlargest(10).index.tolist()

df_train_eng = clean_and_engineer_train(df_train, top_states, top_protos)

valid_num = [c for c in NUM_COLS if c in df_train_eng.columns]
for col in valid_num: 
    df_train_eng[col] = np.log1p(df_train_eng[col].clip(lower=0))

current_bin_cols = [c for c in df_train_eng.columns if c.startswith('is_dport_') or c in BIN_COLS]

print("[*] Skalowanie (RobustScaler) oraz kodowanie (OHE)...")
transformer = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), valid_num),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['State', 'Proto']),
        ('bin', 'passthrough', current_bin_cols) 
    ], remainder='drop'
)

X_train_matrix = transformer.fit_transform(df_train_eng)

if np.isnan(X_train_matrix).any() or np.isinf(X_train_matrix).any():
    X_train_matrix = np.nan_to_num(X_train_matrix)

print(f"[+] Dane przygotowane! Kształt macierzy X_train_matrix: {X_train_matrix.shape}")

>>> KROK 1: ŁADOWANIE PRÓBKI DANYCH I INŻYNIERIA CECH <<<
[*] Pobieranie próbek z 14 plików...
[*] Konwersja czasu i sortowanie...
[*] Generowanie profili behawioralnych...
[*] Skalowanie (RobustScaler) oraz kodowanie (OHE)...
[+] Dane przygotowane! Kształt macierzy X_train_matrix: (28000, 88)


# Dostrajanie hiperparametrów

In [2]:
import itertools
import time
import pandas as pd
import numpy as np
import umap.umap_ as umap
import hdbscan
from sklearn.metrics import silhouette_score
import warnings

# Ignorujemy ostrzeżenia
warnings.filterwarnings('ignore')

print("="*90)
print(">>> AUTOMATYCZNE DOSTRAJANIE HIPERPARAMETRÓW (GRID SEARCH) <<<")
print("="*90)

# ========================================================
# --- SZYBKIE CIĘCIE MACIERZY (TARGET_SAMPLE) ---
# ========================================================
TARGET_SAMPLE = 10000

if len(X_train_matrix) > TARGET_SAMPLE:
    np.random.seed(42)
    idx = np.random.choice(len(X_train_matrix), TARGET_SAMPLE, replace=False)
    X_train_sample = X_train_matrix[idx]
else:
    X_train_sample = X_train_matrix

print(f"[*] Rozmiar macierzy do testów (po redukcji): {X_train_sample.shape}")
# ========================================================

# 1. Definicja przestrzeni poszukiwań (Grid)
param_grid = {
    'umap_neighbors': [15, 30, 50],
    'umap_components': [3, 5, 10], 
    'hdb_min_cluster_size': [50, 100, 200],
    'hdb_min_samples': [15, 30]
}

# Generowanie wszystkich kombinacji
keys = param_grid.keys()
combinations = list(itertools.product(*param_grid.values()))
print(f"[*] Do przetestowania: {len(combinations)} kombinacji.")
print("[*] Proszę czekać, to może potrwać kilka minut...\n")

results = []
total_start = time.time()

# 2. Główna pętla testująca
for idx, values in enumerate(combinations):
    params = dict(zip(keys, values))
    
    # Drukujemy postęp
    print(f"\rPrzetwarzanie kombinacji {idx+1}/{len(combinations)}...", end="")
    
    try:
        # KROK A: Redukcja UMAP
        reducer = umap.UMAP(
            n_neighbors=params['umap_neighbors'],
            n_components=params['umap_components'],
            min_dist=0.0,
            metric='euclidean',
            random_state=42,
            n_jobs=1,
            init='random' # <--- Tu naprawiono brakujący przecinek wyżej!
        )
        
        # UWAGA: Używamy uciętej macierzy X_train_sample!
        X_umap = reducer.fit_transform(X_train_sample)
        
        # KROK B: Klastrowanie HDBSCAN
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=params['hdb_min_cluster_size'],
            min_samples=params['hdb_min_samples'],
            metric='euclidean',
            cluster_selection_method='eom',
            gen_min_span_tree=True, # <--- KRYTYCZNE DLA DBCV
            core_dist_n_jobs=1
        )
        labels = clusterer.fit_predict(X_umap)
        
        # KROK C: Wyliczanie metryk
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_points = list(labels).count(-1)
        noise_pct = (noise_points / len(labels)) * 100
        
        # Ekstrakcja metryki DBCV
        dbcv_score = getattr(clusterer, 'relative_validity_', np.nan)
        
        # Silhouette na podzbiorze bez szumu
        mask = labels != -1
        if n_clusters > 1 and mask.sum() > 1000:
            idx_sil = np.random.choice(np.where(mask)[0], min(5000, mask.sum()), replace=False)
            sil_score = silhouette_score(X_umap[idx_sil], labels[idx_sil])
        else:
            sil_score = np.nan
            
        # Zapis wyników
        results.append({
            'Neighbors': params['umap_neighbors'],
            'Components': params['umap_components'],
            'Min_Clust': params['hdb_min_cluster_size'],
            'Min_Samp': params['hdb_min_samples'],
            'Klastry': n_clusters,
            'Szum (%)': round(noise_pct, 1),
            'DBCV Score': dbcv_score,
            'Silhouette': sil_score
        })
        
    except Exception as e:
        print(f"\n[!] Błąd przy kombinacji {params}: {e}")
        continue

print(f"\n\n[*] Zakończono Grid Search w czasie {time.time() - total_start:.1f} s.")

# 3. Tworzenie i formatowanie tabeli wyników
df_results = pd.DataFrame(results)

df_valid = df_results[(df_results['Klastry'] > 3) & (df_results['Szum (%)'] < 80)].copy()
df_valid.sort_values(by='DBCV Score', ascending=False, inplace=True)

print("\n" + "="*115)
print(f"{'TOP 10 KOMBINACJI (POSORTOWANE WZGLĘDEM DBCV SCORE)':^115}")
print("="*115)
print(df_valid.head(10).to_string(index=False, na_rep='N/A', float_format="%.4f"))
print("="*115)

>>> AUTOMATYCZNE DOSTRAJANIE HIPERPARAMETRÓW (GRID SEARCH) <<<
[*] Rozmiar macierzy do testów (po redukcji): (10000, 88)
[*] Do przetestowania: 54 kombinacji.
[*] Proszę czekać, to może potrwać kilka minut...

Przetwarzanie kombinacji 54/54...

[*] Zakończono Grid Search w czasie 4064.2 s.

                                TOP 10 KOMBINACJI (POSORTOWANE WZGLĘDEM DBCV SCORE)                                
 Neighbors  Components  Min_Clust  Min_Samp  Klastry  Szum (%)  DBCV Score  Silhouette
        50          10         50        30       67    4.9000      0.7249      0.8220
        50          10        100        15       35   18.9000      0.6733      0.8280
        50          10         50        15       76    3.5000      0.6600      0.8174
        50           3         50        30       71    3.0000      0.6477      0.7426
        50           5        100        30       35   13.8000      0.6289      0.7349
        50           3        100        30       38   11.9000      0.